# codingStandard ML/DL — Colab clean-runtime validation
Run this notebook top-to-bottom in a fresh Colab runtime.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
print('Python:', sys.version)
print('CWD:', ROOT)


In [ ]:
%pip install -q psutil
print('psutil ready')


In [ ]:
import json
import subprocess
result = subprocess.run([sys.executable, 'platform/colab/validate_runtime.py', '--json'], check=True, capture_output=True, text=True)
report = json.loads(result.stdout)
print(json.dumps(report, indent=2, sort_keys=True))


In [ ]:
subprocess.run([sys.executable, 'scripts/validation/validate_agent_routing.py'], check=True)


In [ ]:
import importlib.util
if importlib.util.find_spec('torch') is None:
    print('PyTorch is not installed; skipping framework smoke test.')
else:
    import torch
    device = 'cuda' if torch.cuda.is_available() else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu')
    x = torch.randn(4, 8, device=device)
    layer = torch.nn.Linear(8, 2, device=device)
    loss = layer(x).square().mean()
    loss.backward()
    print('PyTorch smoke passed:', device, 'torch', torch.__version__)


In [ ]:
if importlib.util.find_spec('torch') is None:
    print('PyTorch unavailable; checkpoint restore test skipped.')
else:
    import torch
    durable = Path('/content/codingstandard-validation')
    durable.mkdir(parents=True, exist_ok=True)
    path = durable / 'checkpoint.pt'
    state = {'step': 1, 'tensor': torch.arange(4)}
    torch.save(state, path)
    restored = torch.load(path, map_location='cpu', weights_only=False)
    assert restored['step'] == 1
    assert restored['tensor'].tolist() == [0, 1, 2, 3]
    print('checkpoint/restore passed:', path)
